# ICS 591 Team 2 — Stage 1b: ViT-S/16 vs ConvNeXt-Tiny tie-breaker

Stage 1 showed ViT-S/16 and ConvNeXt-Tiny are statistically tied on macro-F1 (overlapping 95% CIs), while both clearly beat EfficientNet-B0, ResNet-50 and MobileNetV3. This notebook settles the choice between the two.

**What changes from Stage 1:** only the *training* random seed. The data split, hyperparameters and test set are identical (split seed stays 42), so any difference comes from the models, not the data.

**Runs:** 3 training seeds per model (42, 7, 123). Seed-42 models are reloaded from Stage 1 checkpoints in Drive, so only 4 new trainings are needed, about 20 minutes on a T4.

**Also added:** temperature scaling. Stage 1's reliability diagram showed both models are *underconfident*, so we fit one temperature per model on validation data and re-measure calibration. This is the calibration fix we will carry into Stage 2.

**Decision rule, fixed before seeing results** (so the choice isn't cherry-picked):
1. If one model is better on **both** mean macro-F1 and mean false-accept rate → choose it.
2. Otherwise → choose the model with the **lower mean false-accept rate**, because a false accept is the costlier error in our system (fraudulent payout / contaminated material to a recycler; see C2).
3. If false-accept rates are also within 0.5 percentage points → choose the **smaller model** (cheaper to deploy and fine-tune).

**How to run:** Runtime → T4 GPU → Run all. Use the same Google account as Stage 1 so it can find the saved checkpoints.

In [ ]:
# 1. Setup
!pip -q install timm
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE - go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# 2. Configuration — identical to Stage 1 except the list of training seeds
DATASET = 'realwaste'
SAVE_TO_DRIVE = True

MODELS = {
    'ViT-S/16':      'vit_small_patch16_224.augreg_in21k_ft_in1k',
    'ConvNeXt-Tiny': 'convnext_tiny.fb_in22k_ft_in1k',
}
TRAIN_SEEDS = [42, 7, 123]

EPOCHS       = 12
BATCH_SIZE   = 32
LR           = 2e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTH = 0.1
IMG_SIZE     = 224
SEED         = 42        # SPLIT seed — must stay 42 so the test set is the same as Stage 1
NUM_WORKERS  = 2
SPLIT        = (0.70, 0.15, 0.15)

TARGET_PLASTIC_RECALL = 0.95
REVIEW_CONF_THRESHOLD = 0.70

In [ ]:
# 3. Output folder
import pathlib
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = pathlib.Path('/content/drive/MyDrive/ics591_model_selection') / DATASET
else:
    OUT = pathlib.Path('/content/outputs') / DATASET
OUT.mkdir(parents=True, exist_ok=True)
print('Saving results to', OUT)

In [ ]:
# 4. Download and locate the dataset
import zipfile, urllib.request

URLS = {
    'realwaste': 'https://archive.ics.uci.edu/static/public/908/realwaste.zip',
    'trashnet':  'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip',
}
RAW = pathlib.Path('/content/raw'); RAW.mkdir(parents=True, exist_ok=True)
zip_path = RAW / f'{DATASET}.zip'
extract_dir = RAW / DATASET

if not zip_path.exists():
    print('Downloading', URLS[DATASET], '...')
    urllib.request.urlretrieve(URLS[DATASET], zip_path)
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for _ in range(2):  # some archives contain zips inside the zip
        for inner in list(extract_dir.rglob('*.zip')):
            with zipfile.ZipFile(inner) as z:
                z.extractall(inner.parent)
            inner.unlink()

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def has_images(d):
    return any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file())

# find the folder whose sub-folders are the classes
best, best_n = None, 0
for d in [extract_dir, *[p for p in extract_dir.rglob('*') if p.is_dir()]]:
    if '__MACOSX' in d.parts:
        continue
    n = sum(1 for c in d.iterdir() if c.is_dir() and has_images(c))
    if n > best_n:
        best, best_n = d, n
assert best_n >= 2, 'Could not find class folders - inspect /content/raw in the file browser'
SRC_ROOT = best
CLASSES = sorted(c.name for c in SRC_ROOT.iterdir() if c.is_dir() and has_images(c))
print('Class folder root:', SRC_ROOT)
print(len(CLASSES), 'classes:', CLASSES)

In [ ]:
# 5. Pre-resize images once (short side 256px) so training isn't bottlenecked by decoding
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

DST_ROOT = pathlib.Path('/content/data') / DATASET
jobs = []
for ci, cname in enumerate(CLASSES):
    (DST_ROOT / cname).mkdir(parents=True, exist_ok=True)
    for f in sorted((SRC_ROOT / cname).iterdir()):
        if f.suffix.lower() in IMG_EXT:
            jobs.append((f, DST_ROOT / cname / (f.stem + '.jpg'), ci))

def resize_one(job):
    src, dst, ci = job
    if dst.exists():
        return (str(dst), ci)
    try:
        im = Image.open(src).convert('RGB')
        s = 256 / min(im.size)
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
        im.save(dst, quality=95)
        return (str(dst), ci)
    except Exception as e:
        print('Skipping unreadable image', src, e)
        return None

with ThreadPoolExecutor(8) as ex:
    SAMPLES = [r for r in ex.map(resize_one, jobs) if r is not None]
print(len(SAMPLES), 'images ready')

In [ ]:
# 6. Stratified train / validation / test split (same split for every model)
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

paths = np.array([p for p, _ in SAMPLES]); labels = np.array([y for _, y in SAMPLES])
tr_p, tmp_p, tr_y, tmp_y = train_test_split(paths, labels, test_size=1 - SPLIT[0],
                                            stratify=labels, random_state=SEED)
rel_test = SPLIT[2] / (SPLIT[1] + SPLIT[2])
va_p, te_p, va_y, te_y = train_test_split(tmp_p, tmp_y, test_size=rel_test,
                                          stratify=tmp_y, random_state=SEED)
TRAIN, VAL, TEST = list(zip(tr_p, tr_y)), list(zip(va_p, va_y)), list(zip(te_p, te_y))

counts = pd.DataFrame({
    'train': np.bincount(tr_y, minlength=len(CLASSES)),
    'val':   np.bincount(va_y, minlength=len(CLASSES)),
    'test':  np.bincount(te_y, minlength=len(CLASSES)),
}, index=CLASSES)
counts.loc['TOTAL'] = counts.sum()
counts.to_csv(OUT / 'split_counts.csv')

PLASTIC = next(i for i, c in enumerate(CLASSES) if 'plastic' in c.lower())
print('"Accept" class for the recycling gate:', CLASSES[PLASTIC])
counts

In [ ]:
# 7. Data pipeline and training / evaluation helpers
import math, time, json, gc, random, copy
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class ImgDS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), int(y)

def make_tfms(model):
    cfg = timm.data.resolve_data_config({}, model=model)  # each model's own normalisation
    norm = T.Normalize(cfg['mean'], cfg['std'])
    train_tf = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.ColorJitter(0.3, 0.3, 0.3),   # simulates varied lighting and phone cameras
        T.ToTensor(), norm])
    eval_tf = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE), T.ToTensor(), norm])
    return train_tf, eval_tf

def loaders_for(model):
    train_tf, eval_tf = make_tfms(model)
    g = torch.Generator(); g.manual_seed(SEED)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    return (DataLoader(ImgDS(TRAIN, train_tf), BATCH_SIZE, shuffle=True, drop_last=True, generator=g, **kw),
            DataLoader(ImgDS(VAL, eval_tf), 64, **kw),
            DataLoader(ImgDS(TEST, eval_tf), 64, **kw))

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_one_epoch(model, loader, opt, sched, scaler, crit):
    model.train(); tot, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * x.size(0); n += x.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x.to(DEVICE, non_blocking=True))
        P.append(logits.float().softmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

In [ ]:
# 8. Metrics tied to our system's error costs
def ece_score(probs, y, n_bins=15):
    # Expected Calibration Error: does a 0.8 confidence really mean 80% correct?
    conf, pred = probs.max(1), probs.argmax(1); correct = pred == y
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def plastic_gate(pv, yv, pt, yt):
    # Treat 'plastic' as ACCEPT and everything else as REJECT, like the real verifier.
    # Pick the threshold on VALIDATION so that plastic recall >= target, then measure on TEST.
    t = float(np.quantile(pv[yv == PLASTIC, PLASTIC], 1 - TARGET_PLASTIC_RECALL))
    accept = pt[:, PLASTIC] >= t; is_p = yt == PLASTIC
    return {'gate_threshold': t,
            'far_at_target': float(accept[~is_p].mean()),   # false accept: non-plastic paid out
            'frr_at_target': float(1 - accept[is_p].mean())}  # false reject: honest household turned away

def review_routing(pt, yt):
    conf = pt.max(1); auto = conf >= REVIEW_CONF_THRESHOLD
    return {'auto_decided_share': float(auto.mean()),
            'auto_decided_accuracy': float((pt.argmax(1)[auto] == yt[auto]).mean()) if auto.any() else float('nan')}

def latency_ms(model, device, runs):
    model = model.to(device).eval(); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        for _ in range(5): model(x)
        if device.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(runs): model(x)
        if device.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t) / runs * 1000

In [ ]:
# 9. Check the split is identical to Stage 1
prev = OUT / 'split_counts.csv'
if prev.exists():
    same = pd.read_csv(prev, index_col=0).equals(counts.astype('int64'))
    print('Split identical to Stage 1:', same)
    assert same, 'Split differs from Stage 1 - do not compare against Stage 1 results'
else:
    print('No Stage 1 split file found; results are still internally consistent.')

In [ ]:
# 10. Temperature scaling (fitted on validation, applied to test)
from scipy.optimize import minimize_scalar

def fit_temperature(pv, yv):
    logp = np.log(np.clip(pv, 1e-12, 1))
    def nll(t):
        z = logp / t; z = z - z.max(1, keepdims=True)
        lp = z - np.log(np.exp(z).sum(1, keepdims=True))
        return -lp[np.arange(len(yv)), yv].mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method='bounded').x)

def apply_temperature(p, t):
    z = np.log(np.clip(p, 1e-12, 1)) / t; z = z - z.max(1, keepdims=True)
    e = np.exp(z); return e / e.sum(1, keepdims=True)

In [ ]:
# 11. Train / reload every (model, seed) run — resumes automatically if interrupted
def slug(name): return name.replace('/', '_').replace(' ', '_')

def loaders_for_seed(model, seed):
    train_tf, eval_tf = make_tfms(model)
    g = torch.Generator(); g.manual_seed(seed)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    return (DataLoader(ImgDS(TRAIN, train_tf), BATCH_SIZE, shuffle=True, drop_last=True, generator=g, **kw),
            DataLoader(ImgDS(VAL, eval_tf), 64, **kw),
            DataLoader(ImgDS(TEST, eval_tf), 64, **kw))

def train_model(timm_id, seed, train_dl, val_dl):
    set_seed(seed)
    model = timm.create_model(timm_id, pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total, warm = EPOCHS * len(train_dl), len(train_dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH); scaler = make_scaler()
    best_f1, best_state, best_ep = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_dl, opt, sched, scaler, crit)
        pv, yv = predict(model, val_dl); vf1 = f1_score(yv, pv.argmax(1), average='macro')
        print(f'  ep {ep:2d} | loss {loss:.3f} | val macro-F1 {vf1:.3f}')
        if vf1 > best_f1:
            best_f1, best_ep = vf1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_state, best_ep

RUNS_FILE = OUT / 'tiebreak_runs.json'
runs = json.loads(RUNS_FILE.read_text()) if RUNS_FILE.exists() else {}

for name, timm_id in MODELS.items():
    for seed in TRAIN_SEEDS:
        key = f'{name}|{seed}'
        if key in runs:
            print(f'[{key}] already done - skipping'); continue
        print(f'\n===== {name}, training seed {seed} =====')
        set_seed(seed)
        stage1_ckpt = OUT / f'{slug(name)}.pt'
        ckpt = OUT / (f'{slug(name)}.pt' if seed == 42 else f'{slug(name)}_seed{seed}.pt')
        if seed == 42 and stage1_ckpt.exists():
            print('  reloading Stage 1 checkpoint')
            model = timm.create_model(timm_id, pretrained=False, num_classes=len(CLASSES)).to(DEVICE)
            model.load_state_dict(torch.load(stage1_ckpt, map_location=DEVICE))
            _, val_dl, test_dl = loaders_for_seed(model, seed); best_ep = None
        else:
            tmp = timm.create_model(timm_id, pretrained=False, num_classes=len(CLASSES))
            train_dl, val_dl, test_dl = loaders_for_seed(tmp, seed); del tmp
            model, state, best_ep = train_model(timm_id, seed, train_dl, val_dl)
            torch.save(state, ckpt)

        pv, yv = predict(model, val_dl); pt, yt = predict(model, test_dl)
        T_ = fit_temperature(pv, yv); pt_cal = apply_temperature(pt, T_)
        np.savez(OUT / f'tiebreak_{slug(name)}_seed{seed}_probs.npz', val=pv, yval=yv, test=pt, ytest=yt)

        pred = pt.argmax(1)
        _, _, f1_per, _ = precision_recall_fscore_support(yt, pred, labels=range(len(CLASSES)), zero_division=0)
        runs[key] = {
            'model': name, 'seed': seed, 'best_epoch': best_ep, 'checkpoint': str(ckpt),
            'test_accuracy': float(accuracy_score(yt, pred)),
            'test_macro_f1': float(f1_score(yt, pred, average='macro')),
            'test_plastic_f1': float(f1_per[PLASTIC]),
            **plastic_gate(pv, yv, pt, yt),
            'ece': ece_score(pt, yt),
            'temperature': T_,
            'ece_after_ts': ece_score(pt_cal, yt),
            **{k + '_after_ts': v for k, v in review_routing(pt_cal, yt).items()},
            **review_routing(pt, yt),
        }
        RUNS_FILE.write_text(json.dumps(runs, indent=2))
        r = runs[key]
        print(f"  TEST macro-F1 {r['test_macro_f1']:.3f} | FAR {r['far_at_target']:.3f} | FRR {r['frr_at_target']:.3f} "
              f"| ECE {r['ece']:.3f} -> {r['ece_after_ts']:.3f} after T={T_:.2f}")
        del model; gc.collect(); torch.cuda.empty_cache()
print('\nAll runs done.')

In [ ]:
# 12. Per-run results and mean ± std across seeds
METRICS = ['test_macro_f1', 'test_plastic_f1', 'far_at_target', 'frr_at_target',
           'ece', 'ece_after_ts', 'temperature', 'auto_decided_share', 'auto_decided_share_after_ts']
per_run = pd.DataFrame(runs.values()).set_index(['model', 'seed']).sort_index()
per_run[METRICS].to_csv(OUT / 'tiebreak_per_run.csv')
display(per_run[METRICS].round(4))

summary = per_run[METRICS].groupby(level='model').agg(['mean', 'std'])
summary.to_csv(OUT / 'tiebreak_summary.csv')
fmt = pd.DataFrame({m: summary[m]['mean'].map('{:.4f}'.format) + ' ± ' + summary[m]['std'].map('{:.4f}'.format)
                    for m in METRICS})
fmt

In [ ]:
# 13. Head-to-head on each seed (paired: same seed, same data)
seeds = sorted(set(per_run.index.get_level_values('seed')))
a, b = list(MODELS)
h2h = pd.DataFrame({
    'seed': seeds,
    f'{a} macro-F1': [per_run.loc[(a, s), 'test_macro_f1'] for s in seeds],
    f'{b} macro-F1': [per_run.loc[(b, s), 'test_macro_f1'] for s in seeds],
    f'{a} FAR': [per_run.loc[(a, s), 'far_at_target'] for s in seeds],
    f'{b} FAR': [per_run.loc[(b, s), 'far_at_target'] for s in seeds],
}).set_index('seed')
h2h['F1 winner'] = np.where(h2h[f'{a} macro-F1'] > h2h[f'{b} macro-F1'], a, b)
h2h['FAR winner'] = np.where(h2h[f'{a} FAR'] < h2h[f'{b} FAR'], a, b)
h2h.to_csv(OUT / 'tiebreak_head_to_head.csv')
h2h.round(4)

In [ ]:
# 14. Apply the decision rule written at the top of the notebook
mean = summary.xs('mean', axis=1, level=1)
f1_best  = mean['test_macro_f1'].idxmax()
far_best = mean['far_at_target'].idxmin()
far_gap  = abs(mean.loc[a, 'far_at_target'] - mean.loc[b, 'far_at_target'])
size_mb  = {n: (OUT / f'{slug(n)}.pt').stat().st_size / 1e6 for n in MODELS}

if f1_best == far_best:
    WINNER, reason = f1_best, 'better on both mean macro-F1 and mean false-accept rate (rule 1)'
elif far_gap >= 0.005:
    WINNER, reason = far_best, (f'lower mean false-accept rate by {far_gap*100:.2f} pp, '
                                f'while {f1_best} has higher macro-F1 (rule 2)')
else:
    WINNER = min(size_mb, key=size_mb.get)
    reason = f'false-accept rates within {far_gap*100:.2f} pp, so the smaller model wins (rule 3)'

best_seed = 42   # fixed in advance; choosing the best-scoring seed would be peeking at the test set
selected = {
    'selected_model': WINNER, 'timm_id': MODELS[WINNER], 'reason': reason,
    'checkpoint_for_stage2': runs[f'{WINNER}|{best_seed}']['checkpoint'],
    'checkpoint_seed': int(best_seed),
    'temperature_for_that_checkpoint': runs[f'{WINNER}|{best_seed}']['temperature'],
    'mean_metrics': {k: float(v) for k, v in mean.loc[WINNER].items()},
}
(OUT / 'selected_model.json').write_text(json.dumps(selected, indent=2))
print(f'SELECTED: {WINNER}\nReason: {reason}')
print(f"Stage 2 starts from: {selected['checkpoint_for_stage2']} (seed {best_seed})")

In [ ]:
# 15. Charts for the report
import matplotlib.pyplot as plt
show = [('test_macro_f1', 'Test macro-F1 (higher is better)'),
        ('far_at_target', f'False-accept rate @ {TARGET_PLASTIC_RECALL:.0%} plastic recall (lower)'),
        ('frr_at_target', 'False-reject rate on plastic (lower)'),
        ('ece_after_ts', 'Calibration error after temperature scaling (lower)')]
names = list(MODELS); colors = ['#4C8BF5', '#F5A04C']
fig, ax = plt.subplots(1, 4, figsize=(22, 5))
for axi, (m, title) in zip(ax, show):
    mu = [summary.loc[n, (m, 'mean')] for n in names]; sd = [summary.loc[n, (m, 'std')] for n in names]
    axi.bar(names, mu, yerr=sd, capsize=6, color=colors)
    for i, n in enumerate(names):
        axi.scatter([i] * len(seeds), per_run.loc[n, m], color='k', zorder=3, s=18)
    axi.set_title(title, fontsize=10)
plt.suptitle('ViT-S/16 vs ConvNeXt-Tiny across 3 training seeds (bars = mean ± std, dots = individual seeds)')
plt.tight_layout(); plt.savefig(OUT / 'tiebreak_comparison.png', dpi=150); plt.show()

# reliability diagram before vs after temperature scaling (seed 42)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5)); edges = np.linspace(0, 1, 11)
for axi, n in zip(ax, names):
    d = np.load(OUT / f'tiebreak_{slug(n)}_seed42_probs.npz'); T_ = runs[f'{n}|42']['temperature']
    for p, lab in [(d['test'], 'before'), (apply_temperature(d['test'], T_), f'after (T={T_:.2f})')]:
        conf, corr = p.max(1), p.argmax(1) == d['ytest']; xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            mk = (conf > lo) & (conf <= hi)
            if mk.sum() >= 5: xs.append(conf[mk].mean()); ys.append(corr[mk].mean())
        axi.plot(xs, ys, marker='o', label=f'{lab}, ECE {ece_score(p, d["ytest"]):.3f}')
    axi.plot([0, 1], [0, 1], 'k--', label='perfect'); axi.set_title(f'{n} - temperature scaling (seed 42)')
    axi.set_xlabel('confidence'); axi.set_ylabel('accuracy'); axi.legend()
plt.tight_layout(); plt.savefig(OUT / 'tiebreak_temperature_scaling.png', dpi=150); plt.show()

## Files to bring back
From **My Drive → ics591_model_selection → realwaste**: `tiebreak_per_run.csv`, `tiebreak_summary.csv`, `tiebreak_head_to_head.csv`, `selected_model.json`, and the two `tiebreak_*.png` charts.